In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

### Static Prompt

In [5]:
prompts=[
    {"role":"system" , "content":"You are an AI engineer."},
    {"role":"user" , "content":"What is RAG?"}
]

In [6]:
res=llm.invoke(prompts)
print(res.content)

RAG stands for **Retrieval-Augmented Generation**. It's a powerful technique used to enhance the capabilities of large language models (LLMs) by combining them with an external knowledge source.

Think of it like this:

*   **LLMs are great at generating text:** They can write stories, answer questions, summarize information, and much more, based on the vast amount of data they were trained on.
*   **But LLMs have limitations:**
    *   **Knowledge Cutoff:** Their knowledge is frozen at the time of their last training. They don't know about events or information that occurred after that date.
    *   **Hallucinations:** They can sometimes generate plausible-sounding but factually incorrect information.
    *   **Lack of Specificity:** They might not have access to niche, proprietary, or highly specific information that you need for your task.

**RAG addresses these limitations by:**

1.  **Retrieval:** When a user asks a question or provides a prompt, the RAG system first **retrieves**

## Prompt Templates / Dynamic Prompts

In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [8]:
prompts= ChatPromptTemplate.from_messages([
    {"role":"system" , "content":"Translate the input to {language}"},
    {"role":"user" , "content":"{query}"}
])
prompts

ChatPromptTemplate(input_variables=['language', 'query'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='Translate the input to {language}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})])

In [9]:
final_prompt = prompts.format_messages(language="hindi" , query="I want to become an AI Engineer")
final_prompt

[SystemMessage(content='Translate the input to hindi', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I want to become an AI Engineer', additional_kwargs={}, response_metadata={})]

In [10]:
res=llm.invoke(final_prompt)
print(res.content)


मैं एक AI इंजीनियर बनना चाहता हूँ।


### Chains

In [11]:
chains = prompts | llm 

In [12]:
chains

ChatPromptTemplate(input_variables=['language', 'query'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='Translate the input to {language}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='{query}'), additional_kwargs={})])
| ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash-Lite', 'release_date': '2025-06-17', 'last_updated': '2025-06-17', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling'

In [15]:
res=chains.invoke({
    "language":"HINGLISH",
    "query":"I am 21 years old"
})

In [16]:
print(res.content)

Main 21 saal ka hoon.


### Use StrOutputParser

In [20]:
out = StrOutputParser()

In [21]:
chains = prompts | llm | out

In [22]:
res = chains.invoke({
    "language":"HINGLISH",
    "query":"I am 21 yeas old"
})

In [26]:
res

'Main 21 saal ka/ki hoon.'

In [37]:
def transform_case(result):
    return result.upper()

In [38]:
chains = prompts|llm|out|transform_case

In [39]:
res=chains.invoke({
    "language":"HINGLISH",
    "query":"I am 21 yeas old"
})
res

'MAIN 21 SAAL KA HOON.'